# Wikipedia RfA Network Analysis — Project Webpage
Run all cells to generate `index.html` in the current directory.
Commit and push `index.html` to your GitHub Pages repo to publish.

In [ ]:
# ── Configuration — edit these before generating ──────────────────────────

EXPLAINER_NOTEBOOK_URL = "https://nbviewer.org/github/YOUR_USERNAME/YOUR_REPO/blob/main/explainer.ipynb"

# Direct download links to your data files on GitHub
# Replace YOUR_USERNAME/YOUR_REPO with your actual GitHub path
DATA_DOWNLOADS = [
    {
        "label": "Raw RfA voting data (wiki-RfA.txt)",
        "url":   "https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/wiki-RfA.txt",
        "description": "Original dataset: 198,275 voting records in plain text format."
    },
    {
        "label": "Agreement graph (JSON)",
        "url":   "https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/rfa_projected_graph_agree.json",
        "description": "Voter–voter agreement projection with edge weights."
    },
    {
        "label": "Disagreement graph (JSON)",
        "url":   "https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/rfa_projected_graph_disagree.json",
        "description": "Voter–voter disagreement projection with edge weights."
    },
    {
        "label": "Agreement graph with communities (JSON)",
        "url":   "https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/rfa_agree_graph_with_communities.json",
        "description": "Agreement graph with Louvain community labels attached to each node."
    },
]

# Paths to your saved images — relative to where index.html will live
# Leave the src values as-is if images are in the same folder
IMAGES = {
    "degree_distributions":       "degree_distributions.png",
    "weight_distributions":       "weight_distributions.png",
    "top_hubs":                   "top_hubs.png",
    "community_summary":          "community_summary.png",
    "agree_subgraph":             "agree_subgraph.png",
    "disagree_subgraph":          "disagree_subgraph.png",
    "cross_community_conflict":   "cross_community_conflict.png",
    "temporal_activity":          "temporal_activity.png",
    "temporal_support_ratio":     "temporal_support_ratio.png",
    "temporal_success_rate":      "temporal_success_rate.png",
    "temporal_community_activity":"temporal_community_activity.png",
    "temporal_community_support": "temporal_community_support.png",
}

print("Configuration loaded.")

In [ ]:
# ── HTML generator ────────────────────────────────────────────────────────

def make_download_cards(downloads):
    cards = ""
    for d in downloads:
        cards += f"""
        <div class="download-card">
          <div class="download-icon">&#8659;</div>
          <div>
            <a href="{d['url']}" download class="download-link">{d['label']}</a>
            <p class="download-desc">{d['description']}</p>
          </div>
        </div>"""
    return cards


def make_figure(img_key, caption):
    src = IMAGES.get(img_key, "")
    return f"""
    <figure class="figure">
      <img src="{src}" alt="{caption}" loading="lazy">
      <figcaption>{caption}</figcaption>
    </figure>"""


download_cards_html = make_download_cards(DATA_DOWNLOADS)

html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Wikipedia RfA Network Analysis</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link href="https://fonts.googleapis.com/css2?family=Lora:ital,wght@0,400;0,600;1,400&family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;500&display=swap" rel="stylesheet">
  <style>
    :root {{
      --bg:        #f7f4ef;
      --surface:   #ffffff;
      --border:    #d9d3c8;
      --ink:       #1a1a18;
      --ink-soft:  #5a5750;
      --accent:    #2a52a0;
      --accent2:   #b03a2e;
      --mono:      'IBM Plex Mono', monospace;
      --sans:      'IBM Plex Sans', sans-serif;
      --serif:     'Lora', serif;
    }}
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}

    body {{
      background: var(--bg);
      color: var(--ink);
      font-family: var(--sans);
      font-size: 16px;
      line-height: 1.7;
    }}

    /* ── Header ── */
    header {{
      background: var(--ink);
      color: #fff;
      padding: 64px 48px 48px;
      border-bottom: 4px solid var(--accent);
    }}
    header .eyebrow {{
      font-family: var(--mono);
      font-size: 11px;
      letter-spacing: 0.18em;
      text-transform: uppercase;
      color: #aaa;
      margin-bottom: 16px;
    }}
    header h1 {{
      font-family: var(--serif);
      font-size: clamp(28px, 4vw, 52px);
      font-weight: 600;
      line-height: 1.15;
      max-width: 760px;
    }}
    header .subtitle {{
      margin-top: 16px;
      font-size: 17px;
      color: #ccc;
      font-weight: 300;
      max-width: 640px;
    }}

    /* ── Nav ── */
    nav {{
      background: var(--surface);
      border-bottom: 1px solid var(--border);
      padding: 0 48px;
      position: sticky;
      top: 0;
      z-index: 100;
      display: flex;
      gap: 0;
      overflow-x: auto;
    }}
    nav a {{
      font-family: var(--mono);
      font-size: 12px;
      letter-spacing: 0.08em;
      text-transform: uppercase;
      color: var(--ink-soft);
      text-decoration: none;
      padding: 16px 20px;
      border-bottom: 3px solid transparent;
      white-space: nowrap;
      transition: color 0.15s, border-color 0.15s;
    }}
    nav a:hover {{ color: var(--accent); border-color: var(--accent); }}

    /* ── Layout ── */
    main {{
      max-width: 980px;
      margin: 0 auto;
      padding: 0 32px 80px;
    }}

    section {{
      padding: 64px 0 32px;
      border-bottom: 1px solid var(--border);
    }}
    section:last-of-type {{ border-bottom: none; }}

    .section-label {{
      font-family: var(--mono);
      font-size: 11px;
      letter-spacing: 0.18em;
      text-transform: uppercase;
      color: var(--accent);
      margin-bottom: 12px;
    }}

    h2 {{
      font-family: var(--serif);
      font-size: clamp(22px, 2.5vw, 32px);
      font-weight: 600;
      margin-bottom: 20px;
      line-height: 1.25;
    }}

    h3 {{
      font-family: var(--serif);
      font-size: 20px;
      font-weight: 600;
      margin: 40px 0 12px;
    }}

    p {{ margin-bottom: 16px; color: var(--ink); font-weight: 300; max-width: 720px; }}

    /* ── Stat cards ── */
    .stats-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(180px, 1fr));
      gap: 16px;
      margin: 32px 0;
    }}
    .stat-card {{
      background: var(--surface);
      border: 1px solid var(--border);
      border-top: 3px solid var(--accent);
      padding: 20px 20px 16px;
    }}
    .stat-card .number {{
      font-family: var(--mono);
      font-size: 28px;
      font-weight: 600;
      color: var(--accent);
      line-height: 1;
    }}
    .stat-card .label {{
      font-size: 12px;
      color: var(--ink-soft);
      margin-top: 6px;
      font-family: var(--mono);
      letter-spacing: 0.04em;
    }}

    /* ── Figures ── */
    .figure {{
      margin: 32px 0;
      background: var(--surface);
      border: 1px solid var(--border);
      padding: 24px 24px 16px;
    }}
    .figure img {{
      width: 100%;
      height: auto;
      display: block;
    }}
    figcaption {{
      font-family: var(--mono);
      font-size: 12px;
      color: var(--ink-soft);
      margin-top: 10px;
      letter-spacing: 0.02em;
    }}

    .figure-row {{
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 16px;
      margin: 32px 0;
    }}
    @media (max-width: 640px) {{ .figure-row {{ grid-template-columns: 1fr; }} }}

    /* ── Downloads ── */
    .download-card {{
      display: flex;
      align-items: flex-start;
      gap: 16px;
      background: var(--surface);
      border: 1px solid var(--border);
      padding: 20px 24px;
      margin-bottom: 12px;
      transition: border-color 0.15s;
    }}
    .download-card:hover {{ border-color: var(--accent); }}
    .download-icon {{
      font-size: 22px;
      color: var(--accent);
      line-height: 1;
      flex-shrink: 0;
      margin-top: 2px;
    }}
    .download-link {{
      font-family: var(--mono);
      font-size: 13px;
      font-weight: 600;
      color: var(--accent);
      text-decoration: none;
    }}
    .download-link:hover {{ text-decoration: underline; }}
    .download-desc {{
      font-size: 13px;
      color: var(--ink-soft);
      margin: 4px 0 0;
    }}

    /* ── Explainer link ── */
    .explainer-box {{
      background: var(--ink);
      color: #fff;
      padding: 32px 36px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 24px;
      flex-wrap: wrap;
      margin-top: 40px;
    }}
    .explainer-box p {{
      color: #ccc;
      font-weight: 300;
      margin: 6px 0 0;
      font-size: 14px;
    }}
    .explainer-box strong {{ color: #fff; font-family: var(--serif); font-size: 18px; }}
    .explainer-btn {{
      background: var(--accent);
      color: #fff;
      font-family: var(--mono);
      font-size: 12px;
      letter-spacing: 0.1em;
      text-transform: uppercase;
      padding: 14px 28px;
      text-decoration: none;
      white-space: nowrap;
      transition: background 0.15s;
      flex-shrink: 0;
    }}
    .explainer-btn:hover {{ background: #1e3d7a; }}

    /* ── Footer ── */
    footer {{
      background: var(--ink);
      color: #888;
      font-family: var(--mono);
      font-size: 12px;
      padding: 28px 48px;
      text-align: center;
      letter-spacing: 0.04em;
    }}
  </style>
</head>
<body>

<header>
  <div class="eyebrow">Network Analysis &mdash; DTU Course Project</div>
  <h1>Factions &amp; Fault Lines in Wikipedia's Admin Elections</h1>
  <p class="subtitle">A network analysis of 198,275 votes cast across 3,494 Request for Adminship elections, revealing the hidden community structure among Wikipedia's most active editors.</p>
</header>

<nav>
  <a href="#dataset">Dataset</a>
  <a href="#graphs">Network Analysis</a>
  <a href="#communities">Communities</a>
  <a href="#temporal">Temporal</a>
  <a href="#text">Text Analysis</a>
  <a href="#downloads">Downloads</a>
  <a href="#explainer">Explainer</a>
</nav>

<main>

  <!-- ── DATASET ── -->
  <section id="dataset">
    <div class="section-label">01 &mdash; The Dataset</div>
    <h2>Wikipedia's Request for Adminship</h2>
    <p>
      Wikipedia's Request for Adminship (RfA) is the process by which the community votes on whether to grant an editor administrator privileges — tools such as the ability to delete pages, block users, and protect articles.
      Any established editor may cast a support, oppose, or neutral vote, and leave a written comment explaining their reasoning.
    </p>
    <p>
      The dataset covers elections from 2003 to 2013. Each record captures the voter (<em>SRC</em>), the candidate (<em>TGT</em>), the vote value (+1 support, -1 oppose, 0 neutral), the overall outcome of the election, the year, and the full text comment left by the voter.
    </p>

    <div class="stats-grid">
      <div class="stat-card"><div class="number">198,275</div><div class="label">Total votes</div></div>
      <div class="stat-card"><div class="number">11,381</div><div class="label">Unique editors</div></div>
      <div class="stat-card"><div class="number">3,494</div><div class="label">Unique candidates</div></div>
      <div class="stat-card"><div class="number">2003–2013</div><div class="label">Years covered</div></div>
    </div>

    <p>
      The raw data forms a directed multigraph: nodes are editors, and a directed edge from A to B means A cast a vote on B's adminship candidacy.
      Because some editors ran multiple times and some voters participated in hundreds of elections, the graph is dense and richly weighted.
    </p>
  </section>

  <!-- ── NETWORK ANALYSIS ── -->
  <section id="graphs">
    <div class="section-label">02 &mdash; Network Analysis</div>
    <h2>From Votes to Voter Networks</h2>
    <p>
      Rather than analysing the raw voting graph directly, we projected it into two voter–voter networks by asking: for every pair of editors who both voted on the same candidate, did they agree or disagree?
      This yields an <strong>agreement graph</strong> and a <strong>disagreement graph</strong>, each edge carrying a normalized score between 0 and 1 reflecting how consistently two voters aligned.
      Only pairs with at least 4 shared votes were kept, filtering out low-evidence relationships.
    </p>

    <h3>Degree Distributions</h3>
    <p>Both graphs follow heavy-tailed degree distributions — a small number of editors are highly connected, while most have few connections.</p>
    {make_figure('degree_distributions', 'Figure 1. Degree distributions of the agreement (blue) and disagreement (red) graphs on a log scale.')}

    <h3>Edge Weight Distributions</h3>
    <p>Agreement edges cluster strongly near 1.0 — when editors agree, they tend to agree consistently. Disagreement edges are more spread out, suggesting varied levels of ideological distance.</p>
    {make_figure('weight_distributions', 'Figure 2. Distribution of normalized edge weights. Agreement scores range 0–1; disagreement scores range -1–0.')}

    <h3>Most Connected Editors</h3>
    <p>The top hubs in each graph are strikingly different. Agreement hubs are prolific, consensus-oriented long-term editors. Disagreement hubs — led by Xoloz with nearly 1,000 conflict relationships — are systematic contrarians whose voting patterns oppose a large portion of the active voter base.</p>
    {make_figure('top_hubs', 'Figure 3. Top 20 editors by degree in the agreement graph (left) and disagreement graph (right).')}
  </section>

  <!-- ── COMMUNITIES ── -->
  <section id="communities">
    <div class="section-label">03 &mdash; Community Structure</div>
    <h2>Factions in the Voter Network</h2>
    <p>
      We applied the Louvain algorithm to the agreement graph to detect communities of editors who share similar voting standards.
      Five communities emerged — four large factions of roughly equal size and one smaller cluster.
    </p>

    {make_figure('community_summary', 'Figure 4. Community sizes and shares from Louvain community detection, with average intra-community agreement strength.')}

    <h3>Agreement Subgraph</h3>
    <p>The top 150 editors by degree, coloured by community. Node size reflects degree; edge width reflects agreement strength. Communities pull together in the layout, revealing clear structural separation.</p>
    {make_figure('agree_subgraph', 'Figure 5. Top-150 node agreement subgraph. Node colour = community; edge width ∝ normalized agreement score.')}

    <h3>Disagreement Subgraph</h3>
    <p>The same editors projected onto the disagreement graph. Node colour still reflects agreement community, revealing which communities are in conflict with each other.</p>
    {make_figure('disagree_subgraph', 'Figure 6. Top-120 node disagreement subgraph. Edge colour intensity ∝ disagreement strength.')}

    <h3>Cross-Community Conflict</h3>
    <p>
      The heatmaps below show how disagreement distributes across community pairs. The structure is multipolar: communities 1, 2, and 3 are all in tension with each other, while community 2 shows significant internal disagreement.
      The C0–C2 pair stands out with an average normalized disagreement strength of 0.96 — when these editors clash, they almost always land on opposite sides.
    </p>
    {make_figure('cross_community_conflict', 'Figure 7. Cross-community conflict heatmaps. Left: number of disagreement edges. Right: average normalized disagreement strength.')}
  </section>

  <!-- ── TEMPORAL ── -->
  <section id="temporal">
    <div class="section-label">04 &mdash; Temporal Analysis</div>
    <h2>How the RfA Process Changed Over Time</h2>

    <h3>Overall Activity</h3>
    <p>Voting activity peaked in the mid-to-late 2000s and declined significantly toward 2013, reflecting a well-documented trend of declining Wikipedia editor participation.</p>
    {make_figure('temporal_activity', 'Figure 8. Total votes cast, unique voters, and unique candidates per year.')}

    <h3>Support vs Oppose Ratio</h3>
    <p>The proportion of support votes fluctuated considerably across years, suggesting the community's appetite for promoting new admins was not stable over time.</p>
    {make_figure('temporal_support_ratio', 'Figure 9. Stacked support/oppose vote counts per year (top) and support ratio trend (bottom).')}

    <h3>RfA Success Rate</h3>
    <p>The fraction of RfA candidates who were successfully promoted changed markedly over the dataset's timespan, pointing to shifting community standards for adminship.</p>
    {make_figure('temporal_success_rate', 'Figure 10. Total RfAs attempted and fraction promoted per year.')}

    <h3>Community Activity Over Time</h3>
    <p>Different communities peaked in different years, suggesting they partly reflect cohorts of editors who were most active in distinct eras rather than purely ideological groupings.</p>
    {make_figure('temporal_community_activity', 'Figure 11. Votes cast per year by each community.')}

    <h3>Support Ratio per Community</h3>
    <p>Communities differed not just in when they were active, but in how their support tendencies evolved — some communities became more critical over time, others remained consistently strict.</p>
    {make_figure('temporal_community_support', 'Figure 12. Support ratio per community over time.')}
  </section>

  <!-- ── TEXT ANALYSIS ── -->
  <section id="text">
    <div class="section-label">05 &mdash; Text Analysis</div>
    <h2>What Each Community Talks About</h2>
    <p>
      We applied TF-IDF across the vote comment texts of each community to identify vocabulary that is distinctive to each faction.
      Terms that appear frequently within a community but rarely across others score highest, revealing the concepts and concerns that define each community's identity.
    </p>
    <p>
      <!-- INSERT: Word cloud images for each community here. -->
      <em>Word cloud figures to be inserted here.</em>
    </p>
  </section>

  <!-- ── DOWNLOADS ── -->
  <section id="downloads">
    <div class="section-label">06 &mdash; Data Downloads</div>
    <h2>Explore the Data Yourself</h2>
    <p>All processed graph files are available for download. The raw RfA dataset is provided as-is from the Stanford SNAP collection.</p>
    {download_cards_html}
  </section>

  <!-- ── EXPLAINER ── -->
  <section id="explainer">
    <div class="section-label">07 &mdash; Explainer Notebook</div>
    <h2>Full Technical Details</h2>
    <p>The explainer notebook contains the complete analysis pipeline: data parsing, graph construction, projection methodology, community detection, TF-IDF text analysis, and all visualisation code with commentary.</p>
    <div class="explainer-box">
      <div>
        <strong>Explainer Notebook</strong>
        <p>Full pipeline with code, methodology, and results — hosted on nbviewer.</p>
      </div>
      <a href="{EXPLAINER_NOTEBOOK_URL}" target="_blank" rel="noopener" class="explainer-btn">Open Notebook &rarr;</a>
    </div>
  </section>

</main>

<footer>
  DTU &mdash; Social Graphs and Interactions &mdash; Wikipedia RfA Network Analysis
</footer>

</body>
</html>
"""

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html)

print("index.html written successfully.")
print("Commit and push it to your GitHub Pages repository to publish.")